# Composite Contamination Index — Multi-Criteria Scoring
## Programming Decay | Skill Module: Relational Cartographies
**Author:** Jiajia Jiang | UCL Bartlett B-Pro MArch Urban Design RC18

### Purpose
This script computes a **Composite Contamination Index** for each of the 64 grid cells
by normalising and averaging the six environmental features extracted in
`01_feature_extraction.ipynb`.

### Method
1. Read the six raw feature values from the fishnet feature class attribute table
2. Apply **min-max normalisation** (0–100) to each feature independently
3. Compute the **arithmetic mean** across all six normalised features
4. Write the result back to the attribute table as `Composite_Index`

### Interpretation
Higher Composite_Index values indicate cells with compounded environmental burden —
cells that score consistently high across brownfield density, flood exposure, waste
infrastructure, population pressure, and ecological sensitivity.

The highest-scoring cells (36–50: Cell 37, 50, 36, 26, 17, 18) correspond to the
Dagenham Dock and Thames-side industrial corridor — validating the site selection
for the Programming Decay thesis.

### Prerequisites
- ArcGIS Pro 3.6 with ArcPy and NumPy
- Fishnet feature class with spatial join results already computed

In [11]:
import arcpy
import numpy as np

fc = "CreateFishnet_out_feature_class"

fields = ["Cell_ID", "F1_Brownfield", "sum_Area_SQUAREMETERS", 
          "Polygon_Count", "Polygon_Count_1",
          "sum_Area_SQUAREMETERS_1", "sum_USUALRES", 
          "sum_Area_SQUAREMETERS_12", "SHAPE@AREA"]

rows = []
with arcpy.da.SearchCursor(fc, fields) as cur:
    for row in cur:
        sa = row[8]  # SHAPE@AREA
        f1 = row[1] if row[1] else 0                          # Brownfield count
        f2 = (row[2] / sa * 100) if (row[2] and sa) else 0    # Building coverage %
        f3 = (row[3] if row[3] else 0) + (row[4] if row[4] else 0)  # Waste sites
        f4 = (row[5] / sa * 100) if (row[5] and sa) else 0    # Flood zone %
        f5 = row[6] if row[6] else 0                          # Population
        f6 = (row[7] / sa * 100) if (row[7] and sa) else 0    # Ecological sensitivity %
        rows.append([row[0], f1, f2, f3, f4, f5, f6])

# Min-max normalisation (0-100) per feature
arr = np.array([[r[1], r[2], r[3], r[4], r[5], r[6]] for r in rows])
normed = np.zeros_like(arr)
for i in range(6):
    col = arr[:, i]
    mn, mx = col.min(), col.max()
    if mx > mn:
        normed[:, i] = (col - mn) / (mx - mn) * 100
    else:
        normed[:, i] = 0

# Composite = arithmetic mean of all normalised features
composite = normed.mean(axis=1)

# Write back to attribute table
try:
    arcpy.management.AddField(fc, "Composite_Index", "DOUBLE")
except:
    pass  # Field already exists

with arcpy.da.UpdateCursor(fc, ["Cell_ID", "Composite_Index"]) as cur:
    for row in cur:
        cid = row[0]
        for j, r in enumerate(rows):
            if r[0] == cid:
                row[1] = round(float(composite[j]), 2)
                break
        cur.updateRow(row)

print("Done! Composite_Index written to attribute table")
for j, r in enumerate(rows):
    print(f"  Cell {r[0]:2d}: {composite[j]:.1f}")

Done! Composite_Index written to attribute table
